# Versioning data

## Learning objectives

- Version a data file with DVC: the data goes to a bucket, and a small pointer file goes in Git.
- Make a new data version, rerun the declared steps, and compare the results with `dvc metrics diff`.
- Go back to an old data version and its result, with Git and `dvc checkout`.

## Builds on Exercise 2

- DVC stages are the declared steps from [Exercise 2](../02-workflows/workflows.ipynb), plus versioned data.

## The problem

- The workshop's exercises use one project, a study of Palmer penguins, and add a capability in each exercise.
- The project reports how much heavier male penguins are than females: male mean body mass minus female mean body mass, per species, in grams.
- `data/penguins.csv` holds the 2007 field season, and it's in Git. The 2008 and 2009 seasons are coming.
- Git keeps a full copy of every version of every file in every clone. That's fine for a 15 KB table, and not for a 50 GB dataset.
- Each result needs to point to the exact data it came from, so you can get both back later.

## What you'll do

1. Step 1: move the 2007 data from Git to DVC, push it to the bucket, and commit the pointer.
2. Step 2: take the 2008 and 2009 seasons as a new data version, rerun the pipeline, commit and push.
3. Step 3: go back to the 2007 version and its result, then return to the latest.
4. Run one check.

- The whole exercise takes about 15 minutes.
- Only reading? Each step shows what its commands print.

## The rule

- The data is a pet. Each version has a pointer in Git and a copy in the bucket, and nobody edits a version in place.
- `results/` and DVC's cache, `.dvc/cache`, are cattle. `dvc pull` and `dvc repro` rebuild them.

## Get a terminal

- You read this page and paste commands into a terminal. Nothing runs on the page.
- The output shown here is recorded from a real run of these commands.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-03-dataops?urlpath=lab/workspaces/exercise)**

- Free, with nothing to install. The first start can take a few minutes.
- The terminal is on the left, already in the `project` folder. This page is on the right.
- Copy a command: select it and press Ctrl+C, or Cmd+C on a Mac.
- Paste it in the terminal: Ctrl+Shift+V, or right-click and choose Paste. Cmd+V on a Mac is untested, so right-click if it doesn't work.
- Edit a file: double-click it in the file panel on the far left. Save with Ctrl+S.
- A session is temporary and has 2 GB of memory. Download anything you want to keep.

**Or use your own terminal**

- Optional: [use uv on your own machine](../../uv.md).

- Linux or macOS, on x86_64 or ARM, or Windows through WSL2. On Windows, keep the folder in your WSL2 home, not under `/mnt/c`.
- You need [conda](https://github.com/conda-forge/miniforge). You don't need Jupyter.
- Paste this once:

```bash
git clone --branch binder-03-dataops --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git dataops
cd dataops
conda env create --quiet --file .binder/environment.yml
conda activate repro-dataops
python scripts/setup_project.py
cd project
```

- Every command below runs in that `project` folder.
- To start over, go to the folder above it and run `python scripts/setup_project.py --reset`. This deletes your work.
- conda and DVC may print warnings that are harmless. [Messages you can ignore](../../setup.md#messages-you-can-ignore) lists them.

## The project

- `project` is a Git repository with two commits.
- The first is the end of Exercise 2: `clean.py`, `validate.py`, `report.py`, `params.yaml`, a `Snakefile`, the 2007 data and `metrics.json`, all in Git.
- The second declares the same three steps as DVC stages, in `dvc.yaml`. The data is still in Git.
- `../osn-bucket` stands in for an object storage bucket such as OSN. DVC uses a real bucket the same way, and your keys stay in `--local` config, out of Git.

**The commands that made the second commit.** You don't need to run these. Use them on your own project.

- `dvc stage add` declares a step: what it reads, what it writes, and which section of `params.yaml` it uses.
- `dvc repro` runs whatever is out of date, and records the hash of each input and output in `dvc.lock`.

```text
dvc init
dvc remote add -d storage ../osn-bucket
dvc stage add -n clean -d data/penguins.csv -d clean.py -o results/clean.csv python clean.py
dvc stage add -n validate -d results/clean.csv -d validate.py -o results/validated.csv python validate.py
dvc stage add -n report -d results/validated.csv -d report.py -p report -M metrics.json python report.py
dvc repro
git add .
git commit -m "Declare the same steps as DVC stages"
```

Look at the project:

```bash
git log --format='%h  %s'
cat dvc.yaml
dvc metrics show
```

In [1]:
# Recorded output. You don't need to run this.
import sys

sys.path.insert(0, ".exercise")
from page import reset, run

run("git log --format='%h  %s'", "cat dvc.yaml", "dvc metrics show")

$ git log --format='%h  %s'
c6f5d95  Declare the same steps as DVC stages
9e2aab7  Declare the steps in a Snakefile

$ cat dvc.yaml
stages:
  clean:
    cmd: python clean.py
    deps:
    - clean.py
    - data/penguins.csv
    outs:
    - results/clean.csv
  validate:
    cmd: python validate.py
    deps:
    - results/clean.csv
    - validate.py
    outs:
    - results/validated.csv
  report:
    cmd: python report.py
    deps:
    - report.py
    - results/validated.csv
    params:
    - report
    metrics:
    - metrics.json:
        cache: false

$ dvc metrics show
Path          dimorphism_g_adelie    dimorphism_g_chinstrap    dimorphism_g_gentoo    n_rows
metrics.json  648.9                  250.0                     934.2                  103


## Step 1: put the data under DVC

**Problem:** `data/penguins.csv` is in Git, so every version of it goes into every clone.

**Facts**

- `git rm --cached` stops Git tracking a file, and leaves the file in place.
- `dvc add` copies the file into DVC's cache, and writes a pointer file, `data/penguins.csv.dvc`, with the file's hash. It also adds the data file to `data/.gitignore`.
- Git tracks the pointer. `dvc push` copies the data from the cache to the bucket.
- A Git tag names a commit, so you can come back to it in step 3.
- DVC prints `To track the changes with git, run: ...`. The commands below do that.

**Steps**

1. Move the data from Git to DVC, commit the pointer, tag the commit and push the data.

```bash
git rm --cached data/penguins.csv
dvc add data/penguins.csv
git add data/penguins.csv.dvc data/.gitignore
git commit -m "Track the 2007 season with DVC"
git tag season-2007
dvc push
```

In [2]:
# Recorded output. You don't need to run this.
run("git rm --cached data/penguins.csv", "dvc add data/penguins.csv", "git add data/penguins.csv.dvc data/.gitignore", "git commit -m \"Track the 2007 season with DVC\"", "git tag season-2007", "dvc push")

$ git rm --cached data/penguins.csv
rm 'data/penguins.csv'

$ dvc add data/penguins.csv

To track the changes with git, run:

	git add data/penguins.csv.dvc data/.gitignore

To enable auto staging, run:

	dvc config core.autostage true

$ git add data/penguins.csv.dvc data/.gitignore

$ git commit -m "Track the 2007 season with DVC"
[main 49864c3] Track the 2007 season with DVC
 3 files changed, 6 insertions(+), 111 deletions(-)
 create mode 100644 data/.gitignore
 delete mode 100644 data/penguins.csv
 create mode 100644 data/penguins.csv.dvc

$ git tag season-2007

$ dvc push
2 files pushed


- `dvc push` sent two files: the data, and the pipeline's cached output. `results/clean.csv` and `results/validated.csv` have the same contents here, so DVC stores one copy.

2. Look at the pointer, what Git tracks in `data`, and the bucket.

```bash
cat data/penguins.csv.dvc
git ls-files data
find ../osn-bucket -type f
```

In [3]:
# Recorded output. You don't need to run this.
run("cat data/penguins.csv.dvc", "git ls-files data", "find ../osn-bucket -type f")

$ cat data/penguins.csv.dvc
outs:
- md5: 0f65687151ae03322b262c13b51d2fdc
  size: 4934
  hash: md5
  path: penguins.csv

$ git ls-files data
data/.gitignore
data/penguins.csv.dvc

$ find ../osn-bucket -type f
../osn-bucket/files/md5/57/cb81982bab276d65e5b1a25767426e
../osn-bucket/files/md5/0f/65687151ae03322b262c13b51d2fdc


- The pointer holds the file's MD5 hash and size. The bucket stores the file under that hash.
- The two earlier commits still hold the 2007 file in Git. From this commit on, new versions go to the bucket.

## Step 2: a new data version

**Problem:** the 2008 and 2009 seasons arrive, and the result has to come from them.

**Facts**

- `new_season.py` replaces `data/penguins.csv` with all three seasons, 344 penguins.
- `dvc add` stores the new version in the cache, and updates the hash in the pointer. The old version stays in the cache and the bucket.
- `dvc repro` sees that the data for `clean` changed, and reruns `clean` and every stage after it.
- `dvc metrics diff` compares `metrics.json` with your last commit.

**Steps**

1. Take the new seasons, update the pointer, rerun the pipeline and compare.

```bash
python ../scripts/new_season.py
dvc add data/penguins.csv
dvc repro
dvc metrics diff
```

In [4]:
# Recorded output. You don't need to run this.
run("python ../scripts/new_season.py", "dvc add data/penguins.csv", "dvc repro", "dvc metrics diff")

$ python ../scripts/new_season.py
Delivered 344 rows to project/data/penguins.csv, covering 2007 to 2009.

$ dvc add data/penguins.csv

To track the changes with git, run:

	git add data/penguins.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true

$ dvc repro
'data/penguins.csv.dvc' didn't change, skipping
Running stage 'clean':
> python clean.py
clean: kept 333 rows
Updating lock file 'dvc.lock'

Running stage 'validate':
> python validate.py
validate: 333 rows passed
Updating lock file 'dvc.lock'

Running stage 'report':
> python report.py
{"dimorphism_g_adelie": 674.7, "dimorphism_g_chinstrap": 411.8, "dimorphism_g_gentoo": 805.1, "n_rows": 333}
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.

$ dvc metrics diff
Path          Metric                  HEAD    workspace    Change
metrics.json  dimorphism_g_adelie

- With all three seasons, Adelie males are 674.7 g heavier, Chinstrap males 411.8 g and Gentoo males 805.1 g, from 333 penguins.

2. Commit the pointer, the lock and the metrics together, and push the new data.

```bash
git add data/penguins.csv.dvc dvc.lock metrics.json
git commit -m "Add the 2008 and 2009 seasons"
dvc push
```

In [5]:
# Recorded output. You don't need to run this.
run("git add data/penguins.csv.dvc dvc.lock metrics.json", "git commit -m \"Add the 2008 and 2009 seasons\"", "dvc push")

$ git add data/penguins.csv.dvc dvc.lock metrics.json

$ git commit -m "Add the 2008 and 2009 seasons"
[main 27717e7] Add the 2008 and 2009 seasons
 3 files changed, 17 insertions(+), 17 deletions(-)

$ dvc push
2 files pushed


## Step 3: go back to the 2007 version

**Problem:** a reviewer asks for the 2007 numbers, from the 2007 data.

**Facts**

- The tag `season-2007` names the commit from step 1. Its pointer file, `dvc.lock` and `metrics.json` describe the 2007 data and its result.
- `git checkout season-2007 -- <files>` puts those files back as they were in that commit.
- `dvc checkout` makes `data/penguins.csv` and `results/` match the pointer and `dvc.lock`, from the cache. Nothing reruns.
- In a fresh clone, `dvc pull` gets the same files from the bucket.

**Steps**

1. Go back to the 2007 version. The headline is the 2007 one again.

```bash
git checkout season-2007 -- data/penguins.csv.dvc dvc.lock metrics.json
dvc checkout
dvc metrics show
```

In [6]:
# Recorded output. You don't need to run this.
run("git checkout season-2007 -- data/penguins.csv.dvc dvc.lock metrics.json", "dvc checkout", "dvc metrics show")

$ git checkout season-2007 -- data/penguins.csv.dvc dvc.lock metrics.json

$ dvc checkout
M       data/penguins.csv
M       results/clean.csv
M       results/validated.csv

$ dvc metrics show
Path          dimorphism_g_adelie    dimorphism_g_chinstrap    dimorphism_g_gentoo    n_rows
metrics.json  648.9                  250.0                     934.2                  103


2. Return to the latest version. `dvc metrics diff` and `git status --short` print nothing when your files match your last commit.

```bash
git checkout main -- data/penguins.csv.dvc dvc.lock metrics.json
dvc checkout
dvc metrics diff
git status --short
```

In [7]:
# Recorded output. You don't need to run this.
run("git checkout main -- data/penguins.csv.dvc dvc.lock metrics.json", "dvc checkout", "dvc metrics diff", "git status --short")

$ git checkout main -- data/penguins.csv.dvc dvc.lock metrics.json

$ dvc checkout
M       data/penguins.csv
M       results/clean.csv
M       results/validated.csv

$ dvc metrics diff

$ git status --short


## Check your work

- Run this when all three steps are done. Each line says what to do next.
- It clones your last commit into a throwaway folder, with an empty DVC cache.
- It pulls each data version from your bucket, and compares each commit's `metrics.json` with the data it points to. It takes under a minute.

```bash
python ../check.py
```

On the starter project it fails like this:

In [8]:
# Recorded output. You don't need to run this.
reset()
run("python ../check.py")

$ python ../check.py
FAIL  Your last commit has no data/penguins.csv.dvc, so DVC doesn't track the data yet. Do step 1.
FAIL  There's no DVC pointer in your last commit, so a fresh clone has no data to pull. Do step 1.
FAIL  No commit holds a DVC pointer to the 2007 season. Do step 1, and commit it before step 2.
FAIL  Your last commit doesn't hold all three seasons, 2007 to 2009, from the bucket. Do step 2: commit, then dvc push.

0 of 4 parts pass.


## Summary

- DVC keeps the data out of Git, and a pointer to it in Git. The bucket holds every version.
- A commit ties the pointer, `dvc.lock` and `metrics.json` together, so each result points to its data.
- `dvc repro` reruns the declared steps when the data changes, and `dvc metrics diff` shows what moved.
- Git and `dvc checkout` bring back an old version and its result. `dvc pull` does the same in a fresh clone.

## On your own project

- `dvc add` the dataset your last result came from, and `dvc push` it. A local folder works as the remote until you have a bucket.
- Tag the commit that made a published result.
- Clone your project into an empty folder, run `dvc pull`, and check that the result comes back.

## About the data

- Data: palmerpenguins (Horst, Hill and Gorman 2020, doi:10.5281/zenodo.3960218), CC0, from Gorman, Williams and Fraser (2014), PLOS ONE 9(3): e90081. Collected by Palmer Station LTER with NSF support.
- `data/penguins.csv` starts as the 2007 rows, 110 penguins. `scripts/new_season.py` replaces it with the full table, 344 penguins from 2007 to 2009. Both are unchanged from the package.